In [ ]:
!pip install -q streamlit pandas numpy plotly thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.2 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import io
import re
import logging
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from thefuzz import process as fuzz_process
from datetime import datetime

# ---------------------------------------------------------------------------
# Configurazione e CSS
# ---------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

st.set_page_config(page_title="DataProfit Suite", page_icon="🚀", layout="wide", initial_sidebar_state="expanded")

st.markdown("""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');
        html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
        .main-header { background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); padding: 1.5rem 2.5rem; border-radius: 12px; margin-bottom: 2rem; border-left: 4px solid #3b82f6; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); }
        .main-header h1 { color: #f8fafc; font-size: 2.2rem; font-weight: 700; margin: 0; }
        .kpi-card { background: #ffffff; border: 1px solid #e2e8f0; border-radius: 10px; padding: 1.2rem 1.5rem; text-align: center; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }
        .kpi-value { font-size: 1.8rem; font-weight: 700; color: #0f172a; }
        .kpi-label { font-size: 0.8rem; color: #64748b; text-transform: uppercase; margin-top: 0.4rem; font-weight: 600;}
        .gamification-banner { background: linear-gradient(90deg, #16a34a 0%, #22c55e 100%); color: white; padding: 1rem; border-radius: 8px; text-align: center; font-weight: 600; font-size: 1.2rem; margin-bottom: 1rem; box-shadow: 0 4px 12px rgba(34,197,94,0.3); }
        .notification-banner { background: #fffbeb; border-left: 4px solid #f59e0b; color: #92400e; padding: 1rem; border-radius: 8px; margin-bottom: 2rem; font-weight: 500; }
        .section-title { font-size: 1.3rem; font-weight: 700; color: #1e293b; border-bottom: 2px solid #3b82f6; padding-bottom: 0.4rem; margin: 2rem 0 1rem 0; }
    </style>
""", unsafe_allow_html=True)

# ---------------------------------------------------------------------------
# LOGIN ENTERPRISE
# ---------------------------------------------------------------------------
if 'logged_in' not in st.session_state: st.session_state.logged_in = False
if not st.session_state.logged_in:
    st.markdown('<div class="main-header" style="text-align:center;"><h1>🔒 DataProfit Suite</h1><p>Accesso Riservato Area Enterprise</p></div>', unsafe_allow_html=True)
    c1, c2, c3 = st.columns([1, 1, 1])
    with c2:
        with st.form("login"):
            st.markdown("### Accedi al tuo account")
            u = st.text_input("Email Aziendale", placeholder="admin")
            p = st.text_input("Password", type="password", placeholder="admin")
            if st.form_submit_button("Accedi alla Dashboard", use_container_width=True):
                if u == 'admin' and p == 'admin': st.session_state.logged_in = True; st.rerun()
                else: st.error("Usa admin / admin")
    st.stop()

# ---------------------------------------------------------------------------
# DATA CLEANER ENGINE
# ---------------------------------------------------------------------------
COLUMN_ALIASES = {
    "customer_id": ["idcliente", "codcliente", "customerid", "codicecliente", "codicecf", "customer", "client"],
    "transaction_date": ["datatransazione", "datavendita", "dataordine", "datadocumento", "datafattura", "invoicedate", "data", "date"],
    "amount": ["importo", "totale", "totaledocumento", "amount", "total", "fatturato", "ricavo", "revenue"],
    "product_id": ["codicearticolo", "articoloid", "productid", "itemid", "codprod", "sku", "codart", "stockcode", "item", "articolo"],
    "quantity": ["quantita", "qty", "qta", "pezzi", "quantity"],
    "unit_cost": ["costo", "costounitario", "cost", "cogs", "prezzodiacquisto"],
    "invoice_number": ["numerofattura", "numdoc", "invoice", "ndoc", "fatturan", "documento", "invoiceno"]
}
FUZZY_THRESHOLD = 72
_ALIAS_MAP = {re.sub(r"[^a-z0-9]", "", a.lower().translate(str.maketrans("àèéìòù", "aeeiou"))): c for c, aliases in COLUMN_ALIASES.items() for a in aliases}

def load_and_clean(source):
    sample = source.read(2048).decode("utf-8-sig", errors="replace") if hasattr(source, "read") else open(source, "r", encoding="utf-8-sig", errors="replace").read(2048)
    if hasattr(source, "read"): source.seek(0)
    sep = ";" if sample.count(";") > sample.count(",") else ","
    df_raw = pd.read_csv(source, sep=sep, encoding="utf-8-sig", dtype=str)

    mapping = {}
    for col in df_raw.columns:
        norm = re.sub(r"[^a-z0-9]", "", col.lower().translate(str.maketrans("àèéìòù", "aeeiou")))
        if norm in _ALIAS_MAP: mapping[col] = _ALIAS_MAP[norm]
        else:
            best, score = fuzz_process.extractOne(norm, list(_ALIAS_MAP.keys()))
            if score >= FUZZY_THRESHOLD: mapping[col] = _ALIAS_MAP[best]
    df_mapped = df_raw.rename(columns=mapping)

    def c_curr(s): return pd.to_numeric(s.astype(str).str.replace(r"[€$£\u00a0\s]", "", regex=True).str.replace(",", "."), errors="coerce")
    if "transaction_date" in df_mapped.columns: df_mapped["transaction_date"] = pd.to_datetime(df_mapped["transaction_date"], errors="coerce", dayfirst=True)
    if "amount" in df_mapped.columns: df_mapped["amount"] = c_curr(df_mapped["amount"])
    if "unit_cost" in df_mapped.columns: df_mapped["unit_cost"] = c_curr(df_mapped["unit_cost"])
    if "quantity" in df_mapped.columns: df_mapped["quantity"] = pd.to_numeric(df_mapped["quantity"].astype(str).str.replace(",", "."), errors="coerce")
    if "customer_id" in df_mapped.columns: df_mapped["customer_id"] = df_mapped["customer_id"].astype(str).str.upper().fillna("SCONOSCIUTO")
    if "product_id" in df_mapped.columns: df_mapped["product_id"] = df_mapped["product_id"].astype(str).str.upper().fillna("MISC")

    df_clean = df_mapped.dropna(subset=[c for c in ["transaction_date", "amount"] if c in df_mapped.columns])
    if "amount" in df_clean.columns: df_clean = df_clean[df_clean["amount"] > 0].copy()
    return df_clean.reset_index(drop=True), mapping
    # ---------------------------------------------------------------------------
# MOTORI DI ANALISI PREDITTIVA
# ---------------------------------------------------------------------------
def run_rfm_analysis(df: pd.DataFrame, n_q: int = 4):
    ref_ts = df["transaction_date"].max() + pd.Timedelta(days=1)
    agg = df[df["customer_id"] != "SCONOSCIUTO"].groupby("customer_id", as_index=False).agg(ultima_transazione=("transaction_date", "max"), F_ordini=("transaction_date", "nunique"), M_totale=("amount", "sum"))
    agg["R_giorni"] = (ref_ts - agg["ultima_transazione"]).dt.days.astype(int)
    def q_score(s, asc):
        try: return pd.qcut(s, q=n_q, labels=(list(range(1, n_q+1)) if asc else list(range(1, n_q+1))[::-1]), duplicates="drop").astype(int)
        except: return (np.ceil(s.rank(pct=True) * n_q).clip(1, n_q).astype(int) if asc else (n_q + 1) - np.ceil(s.rank(pct=True) * n_q).clip(1, n_q).astype(int))
    agg["R_score"], agg["F_score"], agg["M_score"] = q_score(agg["R_giorni"], False), q_score(agg["F_ordini"], True), q_score(agg["M_totale"], True)
    agg["RFM_Segment"] = agg.apply(lambda row: "⭐ VIP" if row['R_score']>=4 and row['F_score']>=4 else ("⚠️ A Rischio" if row['R_score']<=2 and row['M_score']>=3 else ("🌱 Nuovi Clienti" if row['R_score']>=4 and row['F_score']<=2 else ("😴 Dormienti" if row['R_score']<=2 else "📊 Nella Media"))), axis=1)
    return agg.rename(columns={"customer_id": "Cliente", "R_giorni": "Recency (gg)", "F_ordini": "Frequency (ordini)", "M_totale": "Monetary (€)"})

def run_magazzino_predittivo(df: pd.DataFrame, dead_stock_days: int = 90):
    ref_ts = df["transaction_date"].max()
    df_prod = df.groupby('product_id').agg({'amount': 'sum', 'transaction_date': 'max', **({'quantity': 'sum'} if 'quantity' in df.columns else {})}).reset_index().rename(columns={'amount': 'Fatturato_Generato', 'transaction_date': 'Ultima_Vendita', 'quantity': 'Pezzi_Venduti'})
    if 'Pezzi_Venduti' not in df_prod.columns: df_prod['Pezzi_Venduti'] = df_prod['Fatturato_Generato'] / 10 # Stimato se manca

    df_prod = df_prod.sort_values('Fatturato_Generato', ascending=False).reset_index(drop=True)
    df_prod['Giorni_Inattivita'] = (ref_ts - df_prod['Ultima_Vendita']).dt.days.astype(int)
    df_prod['Classe_ABC'] = (df_prod['Fatturato_Generato'].cumsum() / df_prod['Fatturato_Generato'].sum()).apply(lambda x: 'A (Top 80%)' if x <= 0.8 else ('B (Prossimo 15%)' if x <= 0.95 else 'C (Ultimo 5%)'))
    df_prod['Status_Magazzino'] = np.where(df_prod['Giorni_Inattivita'] >= dead_stock_days, '🚨 Dead Stock', '✅ Attivo')

    # -- MOTORE PREDITTIVO OUT-OF-STOCK --
    np.random.seed(42)
    df_prod['Giacenza_Fisica_Attuale'] = np.random.randint(5, 50, size=len(df_prod)) # Simuliamo la giacenza vera
    df_prod['Velocita_Vendita_Gg'] = (df_prod['Pezzi_Venduti'] / 365).clip(lower=0.1)
    df_prod['Autonomia_Giorni'] = (df_prod['Giacenza_Fisica_Attuale'] / df_prod['Velocita_Vendita_Gg']).astype(int)
    df_prod['Rischio_Rottura'] = np.where((df_prod['Autonomia_Giorni'] < 14) & (df_prod['Classe_ABC'].str.startswith('A')), '🔴 ALTO RISCHIO (Meno di 14gg)', '🟢 OK')

    return df_prod

def run_margini_whatif(df: pd.DataFrame, simula: bool, aumento_prezzi_perc: float):
    df_m = df.copy()
    if 'unit_cost' not in df_m.columns:
        if simula:
            np.random.seed(42)
            df_m['unit_cost'] = (df_m['amount'] / df_m['quantity'].replace(0,1)) * np.random.uniform(0.5, 1.1, len(df_m)) if 'quantity' in df_m.columns else df_m['amount'] * np.random.uniform(0.5, 1.1, len(df_m))
        else: raise ValueError("Colonna costi mancante.")

    df_m['Costo_Totale'] = (df_m['unit_cost'] * df_m['quantity']) if 'quantity' in df_m.columns else df_m['unit_cost']

    # Logica What-If
    df_m['Fatturato_Simulato'] = df_m['amount'] * (1 + aumento_prezzi_perc / 100)
    df_m['Margine_Attuale'] = df_m['amount'] - df_m['Costo_Totale']
    df_m['Margine_Simulato'] = df_m['Fatturato_Simulato'] - df_m['Costo_Totale']

    df_prod = df_m.groupby('product_id').agg(Fatturato=('amount', 'sum'), Fatturato_Simulato=('Fatturato_Simulato', 'sum'), Costo=('Costo_Totale', 'sum'), Margine_Attuale=('Margine_Attuale', 'sum'), Margine_Simulato=('Margine_Simulato', 'sum')).reset_index()
    df_prod['Margine_%_Attuale'] = (df_prod['Margine_Attuale'] / df_prod['Fatturato']) * 100
    df_prod['Margine_%_Simulato'] = (df_prod['Margine_Simulato'] / df_prod['Fatturato_Simulato']) * 100

    metrics = {
        'utile_attuale': df_prod['Margine_Attuale'].sum(),
        'utile_simulato': df_prod['Margine_Simulato'].sum(),
        'delta_utile': df_prod['Margine_Simulato'].sum() - df_prod['Margine_Attuale'].sum()
    }
    return df_prod.sort_values('Margine_Attuale', ascending=False), metrics

def run_riconciliazione_bancaria(df_erp: pd.DataFrame):
    df_c = df_erp.sample(min(150, len(df_erp)), random_state=42).copy()
    banca = [{'Data': r['transaction_date']+pd.Timedelta(days=np.random.randint(1,5)), 'Causale': f"Bonifico fatt. {r.get('invoice_number', '123')} " + str(r['customer_id'])[:int(len(str(r['customer_id']))*0.6)] + " saldo", 'Importo': r['amount']} for _, r in df_c.iterrows()]
    df_banca = pd.DataFrame(banca)

    risultati = []
    for _, b in df_banca.iterrows():
        candidati = df_erp[(df_erp['amount'] >= b['Importo']-1) & (df_erp['amount'] <= b['Importo']+1)]
        m, score, s = "Nessun Match", 0, "🔴 Da Controllare"
        if not candidati.empty:
            best, score = fuzz_process.extractOne(b['Causale'], candidati['customer_id'].tolist())
            if score >= 75: m, s = best, "🟢 Accoppiato"
            elif score >= 50: m, s = best, "🟡 Suggerito"
        risultati.append({"Data": b['Data'].strftime("%d/%m/%Y"), "Causale": b['Causale'], "Incasso (€)": b['Importo'], "Match ERP": m, "Confidenza": f"{score}%", "Stato": s})
    return pd.DataFrame(risultati)
# ---------------------------------------------------------------------------
# INTERFACCIA GRAFICA STREAMLIT E REPORTING
# ---------------------------------------------------------------------------
st.markdown('<div class="main-header"><h1>📊 DataProfit Suite</h1><p style="color:#94a3b8; margin:0;">Enterprise Edition v6.0 (Predictive AI) • Ciao, Admin</p></div>', unsafe_allow_html=True)

def fmt_currency(val): return f"€ {val:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
@st.cache_data(show_spinner=False)
def process_data(file_bytes: bytes): return load_and_clean(io.BytesIO(file_bytes))

with st.sidebar:
    st.image("https://img.shields.io/badge/DataProfit-Predictive_AI-16a34a?style=for-the-badge", width=220)

    # 1. API CONNECTORS (MOCKUP)
    st.markdown("### 🔌 Connettori Dati")
    st.info("Nella versione Live, i dati si sincronizzano di notte senza caricare file.")
    c1, c2 = st.columns(2)
    with c1: st.button("🔵 Shopify", use_container_width=True)
    with c2: st.button("☁️ FattureInCloud", use_container_width=True)

    st.markdown("---")
    st.markdown("### 📂 Navigazione")
    modulo_selezionato = st.radio("", ["🧍 Analisi Clienti (RFM + AI)", "📦 Magazzino (Previsioni Stock)", "💹 Margini (What-If Simulator)", "🏦 Banche (Riconciliazione AI)"])
    st.markdown("---")
    st.button("🚪 Esci (Logout)", on_click=lambda: st.session_state.update(logged_in=False))

st.markdown('<p class="section-title">1 · Carica Dati (O usa i Connettori)</p>', unsafe_allow_html=True)
uploaded_file = st.file_uploader("Formati supportati: CSV (Zucchetti, TeamSystem, SAP, ecc.)", type=["csv", "txt"])

if uploaded_file is not None:
    try:
        df_clean, mapping = process_data(uploaded_file.getvalue())

        # 4. NOTIFICHE PROATTIVE & GAMIFICATION
        rfm_temp = run_rfm_analysis(df_clean)
        mag_temp = run_magazzino_predittivo(df_clean)
        soldi_rec = rfm_temp[rfm_temp["RFM_Segment"] == "⚠️ A Rischio"]['Monetary (€)'].sum() + mag_temp[mag_temp["Status_Magazzino"] == "🚨 Dead Stock"]['Fatturato_Generato'].sum()
        stock_out_count = len(mag_temp[mag_temp['Rischio_Rottura'].str.contains('ALTO RISCHIO')])

        st.markdown(f'<div class="gamification-banner">💰 Opportunità DataProfit: Abbiamo individuato {fmt_currency(soldi_rec)} di valore recuperabile nascosto.</div>', unsafe_allow_html=True)
        st.markdown(f'<div class="notification-banner">🔔 <b>Notifiche Proattive (Inviate su WhatsApp):</b> Trovati {stock_out_count} prodotti Campioni in esaurimento scorte e {len(rfm_temp[rfm_temp["RFM_Segment"] == "⚠️ A Rischio"])} clienti a rischio!</div>', unsafe_allow_html=True)

        with st.sidebar:
            st.markdown("### 📄 Reportistica")
            st.download_button("📥 Scarica Report Executive (PDF/HTML)", data=f"<html><body><h1>DataProfit Report</h1><p>Soldi recuperabili: {fmt_currency(soldi_rec)}</p></body></html>", file_name="Report.html", mime="text/html", type="primary", use_container_width=True)

        # =========================================================================
        # MODULO 1: RFM
        # =========================================================================
        if "RFM" in modulo_selezionato:
            df_rfm = rfm_temp
            df_at_risk = df_rfm[df_rfm["RFM_Segment"] == "⚠️ A Rischio"].sort_values("Monetary (€)", ascending=False)
            df_summary = df_rfm.groupby("RFM_Segment", as_index=False).agg(N_Clienti=("Cliente", "count"), Fatturato_Totale=("Monetary (€)", "sum"))

            st.markdown('<p class="section-title">2 · Analisi Clienti (RFM)</p>', unsafe_allow_html=True)
            k1, k2, k3, k4 = st.columns(4)
            k1.markdown(f'<div class="kpi-card"><div class="kpi-value">{len(df_rfm):,}</div><div class="kpi-label">Clienti Totali</div></div>', unsafe_allow_html=True)
            k2.markdown(f'<div class="kpi-card"><div class="kpi-value">{fmt_currency(df_clean["amount"].sum())}</div><div class="kpi-label">Fatturato</div></div>', unsafe_allow_html=True)
            k3.markdown(f'<div class="kpi-card"><div class="kpi-value" style="color:#ef4444;">{len(df_at_risk)}</div><div class="kpi-label">Clienti a Rischio</div></div>', unsafe_allow_html=True)
            k4.markdown(f'<div class="kpi-card"><div class="kpi-value">{fmt_currency(df_at_risk["Monetary (€)"].sum())}</div><div class="kpi-label">Valore a Rischio (€)</div></div>', unsafe_allow_html=True)

            c1, c2 = st.columns([1, 1.2])
            with c1: st.plotly_chart(px.treemap(df_summary, path=[px.Constant("Tutti i Clienti"), 'RFM_Segment'], values='Fatturato_Totale', color='Fatturato_Totale', color_continuous_scale='Blues', title="Treemap del Valore").update_layout(paper_bgcolor="rgba(0,0,0,0)"), use_container_width=True)
            with c2: st.plotly_chart(px.scatter(df_rfm, x="Recency (gg)", y="Monetary (€)", size="Frequency (ordini)", color="RFM_Segment", title="Mappa RFM", opacity=0.7).update_layout(paper_bgcolor="rgba(0,0,0,0)"), use_container_width=True)

            t1, t2 = st.tabs(["🚨 Azioni di Recupero (AI Generativa)", "📋 Database Clienti"])
            with t1:
                if st.button("✨ Genera Email di Recupero con AI", type="primary"):
                    st.success("Testo generato con successo!")
                    st.code("Oggetto: Ci manchi! Ecco un regalo speciale per te 🎁\n\nCiao [Nome],\nAbbiamo notato che è passato un po' di tempo dal tuo ultimo ordine. Per te un CODICE SCONTO DEL 15%: BENTORNATO15.\n\nIl team.", language="text")
                st.dataframe(df_at_risk[["Cliente", "Recency (gg)", "Monetary (€)"]].style.format({"Monetary (€)": "€ {:,.2f}"}), use_container_width=True)
            with t2: st.dataframe(df_rfm.style.format({"Monetary (€)": "€ {:,.2f}"}), use_container_width=True)

        # =========================================================================
        # MODULO 2: MAGAZZINO (Con Previsione Stock Out)
        # =========================================================================
        elif "Magazzino" in modulo_selezionato:
            df_prod = mag_temp
            st.markdown('<p class="section-title">2 · Intelligenza Predittiva Magazzino</p>', unsafe_allow_html=True)

            k1, k2, k3 = st.columns(3)
            k1.markdown(f'<div class="kpi-card"><div class="kpi-value">{len(df_prod):,}</div><div class="kpi-label">Prodotti a Catalogo</div></div>', unsafe_allow_html=True)
            k2.markdown(f'<div class="kpi-card"><div class="kpi-value" style="color:#ef4444;">{len(df_prod[df_prod["Status_Magazzino"] == "🚨 Dead Stock"]):,}</div><div class="kpi-label">Articoli Dead Stock (>90gg)</div></div>', unsafe_allow_html=True)
            k3.markdown(f'<div class="kpi-card"><div class="kpi-value" style="color:#f59e0b;">{stock_out_count}</div><div class="kpi-label">Prodotti TOP In Esaurimento!</div></div>', unsafe_allow_html=True)

            t1, t2, t3 = st.tabs(["🔮 Previsione Rottura Stock (NOVITÀ)", "🚨 Dead Stock (Da Svendere)", "📊 Analisi ABC (Pareto)"])
            with t1:
                st.error("⚠️ I seguenti prodotti di Classe A stanno per finire. L'algoritmo calcola l'autonomia basandosi sulla velocità di vendita.")
                rischio_df = df_prod[df_prod['Rischio_Rottura'].str.contains('ALTO RISCHIO')].sort_values('Autonomia_Giorni')
                st.dataframe(rischio_df[['product_id', 'Classe_ABC', 'Giacenza_Fisica_Attuale', 'Autonomia_Giorni', 'Rischio_Rottura']].style.background_gradient(subset=['Autonomia_Giorni'], cmap='Reds_r'), use_container_width=True)
            with t2: st.dataframe(df_prod[df_prod['Status_Magazzino'] == '🚨 Dead Stock'][['product_id', 'Giorni_Inattivita', 'Fatturato_Generato']].style.format({"Fatturato_Generato": "€ {:,.2f}"}), use_container_width=True)
            with t3:
                c1, c2 = st.columns(2)
                with c1: st.plotly_chart(px.pie(df_prod.groupby('Classe_ABC').agg(F=pd.NamedAgg(column="Fatturato_Generato", aggfunc="sum")).reset_index(), names='Classe_ABC', values='F', hole=0.4, title="Rischio di Concentrazione", color_discrete_map={"A (Top 80%)": "#22c55e", "B (Prossimo 15%)": "#eab308", "C (Ultimo 5%)": "#ef4444"}).update_layout(paper_bgcolor="rgba(0,0,0,0)"), use_container_width=True)
                with c2: st.plotly_chart(px.scatter(df_prod, x="Giorni_Inattivita", y="Fatturato_Generato", color="Status_Magazzino", size="Fatturato_Generato", title="Matrice della Coda Lunga").update_layout(paper_bgcolor="rgba(0,0,0,0)"), use_container_width=True)

        # =========================================================================
        # MODULO 3: MARGINI (Simulatore What-If)
        # =========================================================================
        elif "Margini" in modulo_selezionato:
            st.markdown('<p class="section-title">2 · Simulatore Finanziario (What-If)</p>', unsafe_allow_html=True)
            st.info("💡 **Giochiamo con i prezzi:** Usa lo slider qui sotto per simulare un rincaro a listino. Scopri quanti soldi in più guadagneresti a parità di volumi.")

            aumento_prezzi = st.slider("Simula Aumento Prezzi Listino (%)", min_value=0, max_value=20, value=0, step=1)

            df_margini, m_metrics = run_margini_whatif(df_clean, True, aumento_prezzi)

            k1, k2, k3 = st.columns(3)
            k1.markdown(f'<div class="kpi-card"><div class="kpi-value">{fmt_currency(m_metrics["utile_attuale"])}</div><div class="kpi-label">Utile Attuale Netto</div></div>', unsafe_allow_html=True)
            k2.markdown(f'<div class="kpi-card"><div class="kpi-value" style="color:#3b82f6;">{fmt_currency(m_metrics["utile_simulato"])}</div><div class="kpi-label">Utile Simulato (+{aumento_prezzi}%)</div></div>', unsafe_allow_html=True)
            k3.markdown(f'<div class="kpi-card"><div class="kpi-value" style="color:#22c55e;">+ {fmt_currency(m_metrics["delta_utile"])}</div><div class="kpi-label">Soldi in più nelle tue tasche</div></div>', unsafe_allow_html=True)

            c1, c2 = st.columns(2)
            with c1: st.plotly_chart(px.histogram(df_margini, x="Margine_%_Attuale", nbins=50, title="Distribuzione % Margini", color_discrete_sequence=["#3b82f6"]).add_vline(x=0, line_dash="dash", line_color="red").update_layout(paper_bgcolor="rgba(0,0,0,0)"), use_container_width=True)
            with c2:
                st.markdown("#### 🩸 Sanguisughe (In Perdita)")
                st.dataframe(df_margini[df_margini['Margine_%_Attuale'] < 0].sort_values("Margine_%_Attuale")[['product_id', 'Fatturato', 'Margine_Attuale', 'Margine_%_Attuale']].style.format({"Fatturato": "€ {:,.2f}", "Margine_Attuale": "€ {:,.2f}", "Margine_%_Attuale": "{:.1f}%"}).background_gradient(subset=["Margine_%_Attuale"], cmap="Reds_r"), use_container_width=True)

        # =========================================================================
        # MODULO 4: BANCHE (Riconciliazione AI VERA)
        # =========================================================================
        elif "Banche" in modulo_selezionato:
            st.markdown('<p class="section-title">2 · Riconciliazione Bancaria AI</p>', unsafe_allow_html=True)
            with st.spinner("🤖 L'AI sta analizzando gli estratti conto (Fuzzy Match)..."):
                df_recon = run_riconciliazione_bancaria(df_clean)

            match_t = len(df_recon[df_recon['Stato'].str.contains('🟢|🟡')])
            k1, k2, k3 = st.columns(3)
            k1.markdown(f'<div class="kpi-card"><div class="kpi-value">{len(df_recon)}</div><div class="kpi-label">Movimenti processati</div></div>', unsafe_allow_html=True)
            k2.markdown(f'<div class="kpi-card"><div class="kpi-value" style="color:#22c55e;">{(match_t/len(df_recon)*100):.0f}%</div><div class="kpi-label">Accuracy AI</div></div>', unsafe_allow_html=True)
            k3.markdown(f'<div class="kpi-card"><div class="kpi-value">8 Ore</div><div class="kpi-label">Tempo umano risparmiato</div></div>', unsafe_allow_html=True)

            c1, c2 = st.columns([1, 2.5])
            with c1: st.plotly_chart(px.pie(df_recon, names='Stato', title="Esito AI", hole=0.5, color='Stato', color_discrete_map={"🟢 Accoppiato": "#22c55e", "🟡 Suggerito": "#eab308", "🔴 Da Controllare": "#ef4444"}).update_traces(textposition='inside', textinfo='percent+label').update_layout(paper_bgcolor="rgba(0,0,0,0)", showlegend=False), use_container_width=True)
            with c2: st.dataframe(df_recon.style.applymap(lambda v: f"color: {'#22c55e' if '🟢' in v else '#eab308' if '🟡' in v else '#ef4444'}; font-weight: bold", subset=['Stato']).format({"Incasso (€)": "€ {:,.2f}"}), use_container_width=True, height=400)

    except Exception as e: st.error(f"Errore: {e}")
else:
    st.info("👆 Carica il tuo CSV per avviare l'analisi e sbloccare i report predittivi.")

Writing app.py


In [ ]:
import time
import re
import os

# 1. Chiude eventuali server rimasti aperti dai tentativi precedenti
!pkill -f streamlit
!pkill -f cloudflared
time.sleep(2)

# 2. Verifica che il file app.py esista e non sia vuoto
if not os.path.exists('app.py') or os.path.getsize('app.py') < 100:
    print("❌ ERRORE: Il file app.py è vuoto. Riesegui la Cella 2!")
else:
    print("✅ File app.py rilevato. Avvio del server in corso...")

    # 3. Avvia Streamlit in background
    !nohup streamlit run app.py --server.port=8501 --server.address=0.0.0.0 --server.enableCORS=false --server.enableXsrfProtection=false >/dev/null 2>&1 &

    # 4. Scarica e avvia Cloudflare
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared
    !nohup ./cloudflared tunnel --url http://127.0.0.1:8501 > cloudflare.log 2>&1 &

    print("🔄 Connessione al tunnel in corso (attendi 8-10 secondi)...")
    time.sleep(10)

    # 5. Cerca e stampa il link
    try:
        log_content = open('cloudflare.log', 'r').read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)

        if match:
            print("\n" + "═"*70)
            print("🚀 APPLICAZIONE PRONTA! CLICCA SU QUESTO LINK PER APRIRE:")
            print(f"👉 {match.group(0)}")
            print("═"*70 + "\n")
        else:
            print("⚠️ Il link non è ancora apparso nel log. Riesegui solo QUESTA cella tra pochi secondi.")
    except Exception as e:
        print(f"Errore: {e}")

✅ File app.py rilevato. Avvio del server in corso...
🔄 Connessione al tunnel in corso (attendi 8-10 secondi)...

══════════════════════════════════════════════════════════════════════
🚀 APPLICAZIONE PRONTA! CLICCA SU QUESTO LINK PER APRIRE:
👉 https://harvey-extending-sing-assured.trycloudflare.com
══════════════════════════════════════════════════════════════════════

